In [31]:
import torch
import copy
import shutil
from torch import nn
from torch.nn import functional as F
from typing import Tuple
from functools import reduce
import os
import sys
import math
import pprint
from itertools import islice
import torch_geometric as pyg
from torch import optim
from torch.utils import data as torch_data
from torch_geometric.utils import scatter
from torch_geometric.nn.conv import MessagePassing
from torch_geometric.data import Data, InMemoryDataset, download_url, extract_zip

device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")

In [11]:
def index_to_mask(index, size):
    index = index.view(-1)
    size = int(index.max()) + 1 if size is None else size
    mask = index.new_zeros(size, dtype=torch.bool)
    mask[index] = True
    return mask

def edge_match(edge_index, query_index):
    base = edge_index.max(dim=1)[0] + 1
    assert reduce(int.__mul__, base.tolist()) < torch.iinfo(torch.long).max
    scale = base.cumprod(0)
    scale = scale[-1] // scale

    edge_hash = (edge_index * scale.unsqueeze(-1)).sum(dim=0)
    edge_hash, order = edge_hash.sort()
    query_hash = (query_index * scale.unsqueeze(-1)).sum(dim=0)

    start = torch.bucketize(query_hash, edge_hash)
    end = torch.bucketize(query_hash, edge_hash, right=True)
    num_match = end - start

    offset = num_match.cumsum(0) - num_match
    range = torch.arange(num_match.sum(), device=edge_index.device)
    range = range + (start - offset).repeat_interleave(num_match)

    return order[range], num_match


def all_negative(data, batch):
    pos_h_index, pos_t_index, pos_r_index = batch.t()
    r_index = pos_r_index.unsqueeze(-1).expand(-1, data.num_nodes)
    all_index = torch.arange(data.num_nodes, device=batch.device)
    h_index, t_index = torch.meshgrid(pos_h_index, all_index, indexing="ij")
    t_batch = torch.stack([h_index, t_index, r_index], dim=-1)
    all_index = torch.arange(data.num_nodes, device=batch.device)
    t_index, h_index = torch.meshgrid(pos_t_index, all_index, indexing="ij")
    h_batch = torch.stack([h_index, t_index, r_index], dim=-1)

    return t_batch, h_batch


def strict_negative_mask(data, batch):
    pos_h_index, pos_t_index, pos_r_index = batch.t()

    edge_index = torch.stack([data.edge_index[0], data.edge_type])
    query_index = torch.stack([pos_h_index, pos_r_index])
    edge_id, num_t_truth = edge_match(edge_index, query_index)
    t_truth_index = data.edge_index[1, edge_id]
    sample_id = torch.arange(len(num_t_truth), device=batch.device).repeat_interleave(num_t_truth)
    t_mask = torch.ones(len(num_t_truth), data.num_nodes, dtype=torch.bool, device=batch.device)
    t_mask[sample_id, t_truth_index] = 0
    t_mask.scatter_(1, pos_t_index.unsqueeze(-1), 0)

    edge_index = torch.stack([data.edge_index[1], data.edge_type])
    query_index = torch.stack([pos_t_index, pos_r_index])
    edge_id, num_h_truth = edge_match(edge_index, query_index)
    h_truth_index = data.edge_index[0, edge_id]
    sample_id = torch.arange(len(num_h_truth), device=batch.device).repeat_interleave(num_h_truth)
    h_mask = torch.ones(len(num_h_truth), data.num_nodes, dtype=torch.bool, device=batch.device)
    h_mask[sample_id, h_truth_index] = 0
    h_mask.scatter_(1, pos_h_index.unsqueeze(-1), 0)

    return t_mask, h_mask


def compute_ranking(pred, target, mask=None):
    pos_pred = pred.gather(-1, target.unsqueeze(-1))
    ranking = torch.sum((pos_pred <= pred) & mask, dim=-1) + 1
    return ranking


In [12]:
def construct_relation_graph(graph):

    edge_index, edge_type = graph.edge_index, graph.edge_type
    num_nodes, num_rels = graph.num_nodes, graph.num_relations
    device = edge_index.device

    Eh = torch.vstack([edge_index[0], edge_type]).T.unique(dim=0) 

    EhT = torch.sparse_coo_tensor(
        torch.flip(Eh, dims=[1]).T,
        torch.ones(Eh.shape[0], device=device),
        (num_rels, num_nodes)
    )
    Eh = torch.sparse_coo_tensor(
        Eh.T,
        torch.ones(Eh.shape[0], device=device),
        (num_nodes, num_rels)
    )
    Et = torch.vstack([edge_index[1], edge_type]).T.unique(dim=0)


    EtT = torch.sparse_coo_tensor(
        torch.flip(Et, dims=[1]).T,
        torch.ones(Et.shape[0], device=device) ,
        (num_rels, num_nodes)
    )
    Et = torch.sparse_coo_tensor(
        Et.T,
        torch.ones(Et.shape[0], device=device),
        (num_nodes, num_rels)
    )

    Ahh = torch.sparse.mm(EhT, Eh).coalesce()
    Att = torch.sparse.mm(EtT, Et).coalesce()
    Aht = torch.sparse.mm(EhT, Et).coalesce()
    Ath = torch.sparse.mm(EtT, Eh).coalesce()

    edge_index_hh = Ahh.indices()
    edge_index_tt = Att.indices()
    edge_index_ht = Aht.indices()
    edge_index_th = Ath.indices()

    edge_type_hh = torch.zeros(edge_index_hh.size(1), dtype=torch.long, device=device)
    edge_type_tt = torch.ones(edge_index_tt.size(1), dtype=torch.long, device=device)
    edge_type_ht = torch.full((edge_index_ht.size(1),), 2, dtype=torch.long, device=device)
    edge_type_th = torch.full((edge_index_th.size(1),), 3, dtype=torch.long, device=device)

    rel_edge_index = torch.cat([edge_index_hh, edge_index_tt, edge_index_ht, edge_index_th], dim=1)
    rel_edge_type = torch.cat([edge_type_hh, edge_type_tt, edge_type_ht, edge_type_th], dim=0)

    rel_graph = Data(
        edge_index=rel_edge_index,
        edge_type=rel_edge_type,
        num_nodes=num_rels  
    )
    
    rel_graph.num_relations = 4

    graph.relation_graph = rel_graph
    return graph

In [ ]:
def test_construct_relation_graph():
    edge_index = torch.tensor([
        [0, 1, 2, 1], 
        [1, 2, 0, 0]  
    ])
    edge_type = torch.tensor([0, 1, 0, 1])
    num_nodes = 3
    num_relations = 2

    graph = Data(edge_index=edge_index, edge_type=edge_type, num_nodes=num_nodes, num_relations=num_relations)

    output_graph = construct_relation_graph(graph)

    assert hasattr(output_graph, 'relation_graph'), \
        "The output graph must have a 'relation_graph' attribute."

    rel_graph = output_graph.relation_graph

    assert isinstance(rel_graph, Data), \
        "relation_graph must be a torch_geometric.data.Data object."

    assert rel_graph.num_nodes == num_relations, \
        "relation_graph.num_nodes must equal graph.num_relations."
    assert rel_graph.num_relations == 4, \
        "relation_graph.num_relations must be 4 (representing the 4 edge types)."

    assert rel_graph.edge_index.dim() == 2 and rel_graph.edge_index.size(0) == 2, \
        "relation_graph.edge_index must be of shape (2, num_edges)."

    num_edges = rel_graph.edge_index.size(1)
    assert rel_graph.edge_type.size(0) == num_edges, \
        "relation_graph.edge_type must have the same number of elements as there are edges."

    assert torch.all((rel_graph.edge_type >= 0) & (rel_graph.edge_type <= 3)), \
        "All edge types in relation_graph must be in the range 0 to 3."

    expected_edge_index = torch.tensor([
        [0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1],
        [0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0]
    ])
    expected_edge_type = torch.tensor([0, 0, 1, 1, 1, 1, 2, 2, 2, 3, 3, 3])
    expected_num_nodes = 2
    expected_num_relations = 4

    assert torch.equal(rel_graph.edge_index, expected_edge_index), "edge_index does not match expected value."
    assert torch.equal(rel_graph.edge_type, expected_edge_type), "edge_type does not match expected value."
    assert rel_graph.num_nodes == expected_num_nodes, "num_nodes does not match expected value."
    assert rel_graph.num_relations == expected_num_relations, "num_relations does not match expected value."

    print("All tests passed.")

test_construct_relation_graph()


In [14]:
class GeneralizedRelationalConv(MessagePassing):
    def __init__(self, input_dim, output_dim, num_relation, project_relations=False):
        super(GeneralizedRelationalConv, self).__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.num_relation = num_relation
        self.project_relations = project_relations
        self.linear = nn.Linear(input_dim * 2, output_dim)
        self.relation = None
        self.layer_norm = nn.LayerNorm(output_dim)
        self.activation = nn.ReLU()

        if not self.project_relations:
            self.relation = nn.Embedding(num_relation, input_dim)
        else:
            self.relation = None
            self.relation_projection = nn.Sequential(
                nn.Linear(input_dim, input_dim),
                nn.ReLU(),
                nn.Linear(input_dim, input_dim)
            )


    def forward(self, input, query, boundary, edge_index, edge_type, size):

        batch_size = len(query)

        if not self.project_relations:
            relation = self.relation.weight.expand(batch_size, -1, -1) 
        else:
            relation = self.relation_projection(self.relation)

        output = self.propagate(input=input, relation=relation, boundary=boundary, edge_index=edge_index,
                                edge_type=edge_type, size=size)
        return output


    def message(self, input_j, relation, boundary, edge_type):
        relation_j = relation.index_select(self.node_dim, edge_type)
        message = input_j * relation_j
        message = torch.cat([message, boundary], dim=self.node_dim)
        return message

    def aggregate(self, input, index, dim_size):
        index = torch.cat([index, torch.arange(dim_size, device=input.device)]) 
        shape = [1] * input.ndim
        shape[self.node_dim] = -1
        output = scatter(input, index, dim=self.node_dim, dim_size=dim_size, reduce="add")
        return output


    def update(self, update, input):
        output = self.linear(torch.cat([input, update], dim=-1))
        output = self.layer_norm(output)
        output = self.activation(output)
        return output

In [15]:
class RelNBFNet(nn.Module):

    def __init__(self, hidden_dims, num_relation=4):
        super().__init__()
        self.dims = hidden_dims
        self.layers = nn.ModuleList()
        for i in range(len(self.dims) - 1):
            self.layers.append(
                GeneralizedRelationalConv(
                    self.dims[i], self.dims[i + 1], num_relation,
                    )
                )

    def bellmanford(self, data, h_index, separate_grad=False):
        batch_size = len(h_index)

        query = torch.ones(h_index.shape[0], self.dims[0], device=h_index.device, dtype=torch.float)
        index = h_index.unsqueeze(-1).expand_as(query)

        boundary = torch.zeros(batch_size, data.num_nodes, self.dims[0], device=h_index.device)

        boundary.scatter_add_(1, index.unsqueeze(1), query.unsqueeze(1))
        size = (data.num_nodes, data.num_nodes)


        hiddens = []
        layer_input = boundary

        for layer in self.layers:
            hidden = layer(layer_input, query, boundary, data.edge_index, data.edge_type, size)
            if hidden.shape == layer_input.shape:
                hidden = hidden + layer_input
            hiddens.append(hidden)
            layer_input = hidden

        node_query = query.unsqueeze(1).expand(-1, data.num_nodes, -1) 

        output = hiddens[-1]

        return {
            "node_feature": output,
        }

    def forward(self, rel_graph, query):

        output = self.bellmanford(rel_graph, h_index=query)["node_feature"]  

        return output

In [16]:
class EntityNBFNet(nn.Module):

    def __init__(self, hidden_dims, num_relation=1):

        super().__init__()
        self.dims = hidden_dims
        self.layers = nn.ModuleList()
        for i in range(len(self.dims) - 1):
            self.layers.append(
                GeneralizedRelationalConv(
                    self.dims[i], self.dims[i + 1], num_relation,
                   project_relations=True)
            )

        feature_dim = hidden_dims[-1] + self.dims[0]
        self.mlp = nn.Sequential()
        mlp = []
        mlp.append(nn.Linear(feature_dim, feature_dim))
        mlp.append(nn.ReLU())
        mlp.append(nn.Linear(feature_dim, 1))
        self.mlp = nn.Sequential(*mlp)


    def bellmanford(self, data, h_index, r_index):
        batch_size = len(r_index)

        query = self.query[torch.arange(batch_size, device=r_index.device), r_index]
        index = h_index.unsqueeze(-1).expand_as(query)

        boundary = torch.zeros(batch_size, data.num_nodes, self.dims[0], device=h_index.device)
        boundary.scatter_add_(1, index.unsqueeze(1), query.unsqueeze(1))

        size = (data.num_nodes, data.num_nodes)
        edge_weight = torch.ones(data.num_edges, device=h_index.device)

        hiddens = []
        layer_input = boundary

        for layer in self.layers:

            hidden = layer(layer_input, query, boundary, data.edge_index, data.edge_type, size)
            if hidden.shape == layer_input.shape:
                hidden = hidden + layer_input
            hiddens.append(hidden)
            layer_input = hidden

        node_query = query.unsqueeze(1).expand(-1, data.num_nodes, -1) 

        output = torch.cat([hiddens[-1], node_query], dim=-1)

        return {
            "node_feature": output,
        }

    def forward(self, data, relation_representations, batch):
        h_index, t_index, r_index = batch.unbind(-1)

        self.query = relation_representations

        for layer in self.layers:
            layer.relation = relation_representations

        shape = h_index.shape
        h_index, t_index, r_index = self.negative_sample_to_tail(h_index, t_index, r_index, num_direct_rel=data.num_relations // 2)
        assert (h_index[:, [0]] == h_index).all()
        assert (r_index[:, [0]] == r_index).all()

        output = self.bellmanford(data, h_index[:, 0], r_index[:, 0])  
        feature = output["node_feature"]
        index = t_index.unsqueeze(-1).expand(-1, -1, feature.shape[-1])
        feature = feature.gather(1, index)  

        score = self.mlp(feature).squeeze(-1)
        return score.view(shape)

    def negative_sample_to_tail(self, h_index, t_index, r_index, num_direct_rel):
        is_t_neg = (h_index == h_index[:, [0]]).all(dim=-1, keepdim=True)
        new_h_index = torch.where(is_t_neg, h_index, t_index)
        new_t_index = torch.where(is_t_neg, t_index, h_index)
        new_r_index = torch.where(is_t_neg, r_index, r_index + num_direct_rel)
        return new_h_index, new_t_index, new_r_index


In [17]:
class Ultra(nn.Module):

    def __init__(self, rel_model_cfg, entity_model_cfg):
        super(Ultra, self).__init__()

        self.relation_model = RelNBFNet(**rel_model_cfg)
        self.entity_model = EntityNBFNet(**entity_model_cfg)


    def forward(self, data, batch):

        query_rels = batch[:, 0, 2]

        relation_representations = self.relation_model(data.relation_graph, query_rels)
        score = self.entity_model(data, relation_representations, batch)

        return score

In [18]:
class GrailInductiveDataset(InMemoryDataset):

    def __init__(self, root, version, transform=None, pre_transform=construct_relation_graph, merge_valid_test=True):
        self.version = version
        assert version in ["v1", "v2", "v3", "v4"]

        self.merge_valid_test = merge_valid_test
        super().__init__(root, transform, pre_transform)
        self.data, self.slices = torch.load(self.processed_paths[0], weights_only=False)

    @property
    def num_relations(self):
        return int(self.data.edge_type.max()) + 1

    @property
    def raw_dir(self):
        return os.path.join(self.root, "grail", self.name, self.version, "raw")

    @property
    def processed_dir(self):
        return os.path.join(self.root, "grail", self.name, self.version, "processed")

    @property
    def processed_file_names(self):
        return "data.pt"

    @property
    def raw_file_names(self):
        return [
            "train_ind.txt", "valid_ind.txt", "test_ind.txt", "train.txt", "valid.txt"
        ]

    def download(self):
        for url, path in zip(self.urls, self.raw_paths):
            download_path = download_url(url % self.version, self.raw_dir)
            os.rename(download_path, path)

    def process(self):
        test_files = self.raw_paths[:3]
        train_files = self.raw_paths[3:]

        inv_train_entity_vocab = {}
        inv_test_entity_vocab = {}
        inv_relation_vocab = {}
        triplets = []
        num_samples = []

        for txt_file in train_files:
            with open(txt_file, "r") as fin:
                num_sample = 0
                for line in fin:
                    h_token, r_token, t_token = line.strip().split("\t")
                    if h_token not in inv_train_entity_vocab:
                        inv_train_entity_vocab[h_token] = len(inv_train_entity_vocab)
                    h = inv_train_entity_vocab[h_token]
                    if r_token not in inv_relation_vocab:
                        inv_relation_vocab[r_token] = len(inv_relation_vocab)
                    r = inv_relation_vocab[r_token]
                    if t_token not in inv_train_entity_vocab:
                        inv_train_entity_vocab[t_token] = len(inv_train_entity_vocab)
                    t = inv_train_entity_vocab[t_token]
                    triplets.append((h, t, r))
                    num_sample += 1
            num_samples.append(num_sample)

        for txt_file in test_files:
            with open(txt_file, "r") as fin:
                num_sample = 0
                for line in fin:
                    h_token, r_token, t_token = line.strip().split("\t")
                    if h_token not in inv_test_entity_vocab:
                        inv_test_entity_vocab[h_token] = len(inv_test_entity_vocab)
                    h = inv_test_entity_vocab[h_token]
                    assert r_token in inv_relation_vocab
                    r = inv_relation_vocab[r_token]
                    if t_token not in inv_test_entity_vocab:
                        inv_test_entity_vocab[t_token] = len(inv_test_entity_vocab)
                    t = inv_test_entity_vocab[t_token]
                    triplets.append((h, t, r))
                    num_sample += 1
            num_samples.append(num_sample)
        triplets = torch.tensor(triplets)

        edge_index = triplets[:, :2].t()
        edge_type = triplets[:, 2]
        num_relations = int(edge_type.max()) + 1

        train_fact_slice = slice(None, sum(num_samples[:1]))
        test_fact_slice = slice(sum(num_samples[:2]), sum(num_samples[:3]))
        train_fact_index = edge_index[:, train_fact_slice]
        train_fact_type = edge_type[train_fact_slice]
        test_fact_index = edge_index[:, test_fact_slice]
        test_fact_type = edge_type[test_fact_slice]

        train_fact_index = torch.cat([train_fact_index, train_fact_index.flip(0)], dim=-1)
        train_fact_type = torch.cat([train_fact_type, train_fact_type + num_relations])
        test_fact_index = torch.cat([test_fact_index, test_fact_index.flip(0)], dim=-1)
        test_fact_type = torch.cat([test_fact_type, test_fact_type + num_relations])

        train_slice = slice(None, sum(num_samples[:1]))
        valid_slice = slice(sum(num_samples[:1]), sum(num_samples[:2]))
        test_slice = slice(sum(num_samples[:3]), sum(num_samples)) if self.merge_valid_test else slice(sum(num_samples[:4]), sum(num_samples))

        train_data = Data(edge_index=train_fact_index, edge_type=train_fact_type, num_nodes=len(inv_train_entity_vocab),
                          target_edge_index=edge_index[:, train_slice], target_edge_type=edge_type[train_slice], num_relations=num_relations*2)
        valid_data = Data(edge_index=train_fact_index, edge_type=train_fact_type, num_nodes=len(inv_train_entity_vocab),
                          target_edge_index=edge_index[:, valid_slice], target_edge_type=edge_type[valid_slice], num_relations=num_relations*2)
        test_data = Data(edge_index=test_fact_index, edge_type=test_fact_type, num_nodes=len(inv_test_entity_vocab),
                         target_edge_index=edge_index[:, test_slice], target_edge_type=edge_type[test_slice], num_relations=num_relations*2)

        if self.pre_transform is not None:
            train_data = self.pre_transform(train_data)
            valid_data = self.pre_transform(valid_data)
            test_data = self.pre_transform(test_data)

        torch.save((self.collate([train_data, valid_data, test_data])), self.processed_paths[0])

    def __repr__(self):
        return "%s(%s)" % (self.name, self.version)

class NELLInductive(GrailInductiveDataset):
    urls = [
        "https://raw.githubusercontent.com/kkteru/grail/master/data/nell_%s_ind/train.txt",
        "https://raw.githubusercontent.com/kkteru/grail/master/data/nell_%s_ind/valid.txt",
        "https://raw.githubusercontent.com/kkteru/grail/master/data/nell_%s_ind/test.txt",
        "https://raw.githubusercontent.com/kkteru/grail/master/data/nell_%s/train.txt",
        "https://raw.githubusercontent.com/kkteru/grail/master/data/nell_%s/valid.txt"
    ]
    name = "IndNELL"

    def __init__(self, root, version):
        super().__init__(root, version)

class InductiveDataset(InMemoryDataset):

    delimiter = None
    valid_on_inf = True  

    def __init__(self, root, version, transform=None, pre_transform=construct_relation_graph, **kwargs):

        self.version = str(version)
        super().__init__(root, transform, pre_transform)
        self.data, self.slices = torch.load(self.processed_paths[0],weights_only=False)

    def download(self):
        for url, path in zip(self.urls, self.raw_paths):
            download_path = download_url(url % self.version, self.raw_dir)
            os.rename(download_path, path)

    def load_file(self, triplet_file, inv_entity_vocab={}, inv_rel_vocab={}):

        triplets = []
        entity_cnt, rel_cnt = len(inv_entity_vocab), len(inv_rel_vocab)

        with open(triplet_file, "r", encoding="utf-8") as fin:
            for l in fin:
                u, r, v = l.split() if self.delimiter is None else l.strip().split(self.delimiter)
                if u not in inv_entity_vocab:
                    inv_entity_vocab[u] = entity_cnt
                    entity_cnt += 1
                if v not in inv_entity_vocab:
                    inv_entity_vocab[v] = entity_cnt
                    entity_cnt += 1
                if r not in inv_rel_vocab:
                    inv_rel_vocab[r] = rel_cnt
                    rel_cnt += 1
                u, r, v = inv_entity_vocab[u], inv_rel_vocab[r], inv_entity_vocab[v]

                triplets.append((u, v, r))

        return {
            "triplets": triplets,
            "num_node": len(inv_entity_vocab), 
            "num_relation": rel_cnt,
            "inv_entity_vocab": inv_entity_vocab,
            "inv_rel_vocab": inv_rel_vocab
        }

    def process(self):

        train_files = self.raw_paths[:4]

        train_res = self.load_file(train_files[0], inv_entity_vocab={}, inv_rel_vocab={})
        inference_res = self.load_file(train_files[1], inv_entity_vocab={}, inv_rel_vocab={})
        valid_res = self.load_file(
            train_files[2],
            inference_res["inv_entity_vocab"] if self.valid_on_inf else train_res["inv_entity_vocab"],
            inference_res["inv_rel_vocab"] if self.valid_on_inf else train_res["inv_rel_vocab"]
        )
        test_res = self.load_file(train_files[3], inference_res["inv_entity_vocab"], inference_res["inv_rel_vocab"])

        num_train_nodes, num_train_rels = train_res["num_node"], train_res["num_relation"]
        inference_num_nodes, inference_num_rels = test_res["num_node"], test_res["num_relation"]

        train_edges, inf_graph, inf_valid_edges, inf_test_edges = train_res["triplets"], inference_res["triplets"], valid_res["triplets"], test_res["triplets"]

        train_target_edges = torch.tensor([[t[0], t[1]] for t in train_edges], dtype=torch.long).t()
        train_target_etypes = torch.tensor([t[2] for t in train_edges])

        train_fact_index = torch.cat([train_target_edges, train_target_edges.flip(0)], dim=1)
        train_fact_type = torch.cat([train_target_etypes, train_target_etypes + num_train_rels])

        inf_edges = torch.tensor([[t[0], t[1]] for t in inf_graph], dtype=torch.long).t()
        inf_edges = torch.cat([inf_edges, inf_edges.flip(0)], dim=1)
        inf_etypes = torch.tensor([t[2] for t in inf_graph])
        inf_etypes = torch.cat([inf_etypes, inf_etypes + inference_num_rels])

        inf_valid_edges = torch.tensor(inf_valid_edges, dtype=torch.long)
        inf_test_edges = torch.tensor(inf_test_edges, dtype=torch.long)

        train_data = Data(edge_index=train_fact_index, edge_type=train_fact_type, num_nodes=num_train_nodes,
                          target_edge_index=train_target_edges, target_edge_type=train_target_etypes, num_relations=num_train_rels*2)
        valid_data = Data(edge_index=inf_edges if self.valid_on_inf else train_fact_index,
                          edge_type=inf_etypes if self.valid_on_inf else train_fact_type,
                          num_nodes=inference_num_nodes if self.valid_on_inf else num_train_nodes,
                          target_edge_index=inf_valid_edges[:, :2].T,
                          target_edge_type=inf_valid_edges[:, 2],
                          num_relations=inference_num_rels*2 if self.valid_on_inf else num_train_rels*2)
        test_data = Data(edge_index=inf_edges, edge_type=inf_etypes, num_nodes=inference_num_nodes,
                         target_edge_index=inf_test_edges[:, :2].T, target_edge_type=inf_test_edges[:, 2], num_relations=inference_num_rels*2)

        if self.pre_transform is not None:
            train_data = self.pre_transform(train_data)
            valid_data = self.pre_transform(valid_data)
            test_data = self.pre_transform(test_data)

        torch.save((self.collate([train_data, valid_data, test_data])), self.processed_paths[0])

    @property
    def num_relations(self):
        return int(self.data.edge_type.max()) + 1

    @property
    def raw_dir(self):
        return os.path.join(self.root, self.name, self.version, "raw")

    @property
    def processed_dir(self):
        return os.path.join(self.root, self.name, self.version, "processed")

    @property
    def raw_file_names(self):
        return [
            "transductive_train.txt", "inference_graph.txt", "inf_valid.txt", "inf_test.txt"
        ]

    @property
    def processed_file_names(self):
        return "data.pt"

    def __repr__(self):
        return "%s(%s)" % (self.name, self.version)


class IngramInductive(InductiveDataset):

    @property
    def raw_dir(self):
        return os.path.join(self.root, "ingram", self.name, self.version, "raw")

    @property
    def processed_dir(self):
        return os.path.join(self.root, "ingram", self.name, self.version, "processed")


class WKIngram(IngramInductive):

    urls = [
        "https://raw.githubusercontent.com/bdi-lab/InGram/master/data/WK-%s/train.txt",
        "https://raw.githubusercontent.com/bdi-lab/InGram/master/data/WK-%s/msg.txt",
        "https://raw.githubusercontent.com/bdi-lab/InGram/master/data/WK-%s/valid.txt",
        "https://raw.githubusercontent.com/bdi-lab/InGram/master/data/WK-%s/test.txt",
    ]
    name = "wk"

dataset_map = {
    "NELLInductive": NELLInductive,
    "WKIngram": WKIngram,
}

def build_dataset(root, dataset_name):
    dataset_name,version = dataset_name.split(":")
    ds_cls = dataset_map[dataset_name]

    dataset = ds_cls(root = root, version = version)

    print("%s dataset" % (f'{dataset_name}({version})'))

    print("#train: %d, #valid: %d, #test: %d" %
                        (dataset[0].target_edge_index.shape[1], dataset[1].target_edge_index.shape[1],
                            dataset[2].target_edge_index.shape[1]))

    return dataset

In [19]:
def experiment(dataset, checkpoint, root):
    torch.manual_seed(1024)

    print(separator)

    dataset = build_dataset(root, dataset)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    train_data = dataset[0].to(device)
    valid_data = dataset[1].to(device)
    test_data = dataset[2].to(device)

    model = Ultra(
        rel_model_cfg=dict(hidden_dims = [64] * 7),
        entity_model_cfg=dict(hidden_dims = [64] * 7),
    )

    state = torch.load(checkpoint, map_location="cpu", weights_only=True)
    model.load_state_dict(state["model"])

    model = model.to(device)

    if "Ingram" in dataset:
        full_inference_edges = torch.cat([valid_data.edge_index, valid_data.target_edge_index, test_data.target_edge_index], dim=1)
        full_inference_etypes = torch.cat([valid_data.edge_type, valid_data.target_edge_type, test_data.target_edge_type])
        test_filtered_data = Data(edge_index=full_inference_edges, edge_type=full_inference_etypes, num_nodes=test_data.num_nodes)
        val_filtered_data = test_filtered_data
    else:
        full_inference_edges = torch.cat([test_data.edge_index, test_data.target_edge_index], dim=1)
        full_inference_etypes = torch.cat([test_data.edge_type, test_data.target_edge_type])
        test_filtered_data = Data(edge_index=full_inference_edges, edge_type=full_inference_etypes, num_nodes=test_data.num_nodes)

        val_filtered_data = Data(
            edge_index=torch.cat([train_data.edge_index, valid_data.target_edge_index], dim=1),
            edge_type=torch.cat([train_data.edge_type, valid_data.target_edge_type])
        )

    print(separator)
    print("Evaluate on test")
    result = run(model, test_data, filtered_data=test_filtered_data, device=device)
    return result


separator = ">" * 30
line = "-" * 30

@torch.no_grad()
def run(model, test_data, device, filtered_data, batch_size=16):
    test_triplets = torch.cat([test_data.target_edge_index, test_data.target_edge_type.unsqueeze(0)]).t()
    test_loader = torch_data.DataLoader(test_triplets, batch_size)

    model.eval()
    rankings = []
    for batch in test_loader:
        t_batch, h_batch = all_negative(test_data, batch)
        t_pred = model(test_data, t_batch)
        h_pred = model(test_data, h_batch)

        t_mask, h_mask = strict_negative_mask(filtered_data, batch)
        pos_h_index, pos_t_index, pos_r_index = batch.t()
        t_ranking = compute_ranking(t_pred, pos_t_index, t_mask)
        h_ranking = compute_ranking(h_pred, pos_h_index, h_mask)

        rankings += [t_ranking, h_ranking]


    all_ranking = torch.cat(rankings)
    mr = (all_ranking.float()).mean()
    mrr = (1 / all_ranking.float()).mean()
    hits_1 = (all_ranking <= 1).float().mean()
    hits_3 = (all_ranking <= 3).float().mean()
    hits_10 = (all_ranking <= 10).float().mean()

    print(f"MR: {mr:.4f}")
    print(f"MRR: {mrr:.4f}")
    print(f"Hits@1: {hits_1:.4f}")
    print(f"Hits@3: {hits_3:.4f}")
    print(f"Hits@10: {hits_10:.4f}")
    return mr, mrr, hits_1, hits_3, hits_10

In [ ]:
datasets = [
    "NELLInductive:v1", "NELLInductive:v2", "NELLInductive:v3", "NELLInductive:v4", "WKIngram:25", "WKIngram:50", "WKIngram:75", "WKIngram:100"
]

root = "./dataset"
checkpoint_path = "./3g_reproduced.pth"

for dataset in datasets:
    print(f"Dataset: {dataset}")
    experiment(dataset, checkpoint_path, root)